# Simulated backend phase-mask test

This notebook exercises the `slmsuite`-based `qtools.slm` simulated backend. It does not open a physical monitor; frames are kept in an in-memory RGB framebuffer.

Install the optional dependency first if needed:

```bash
pip install "qtools[slm-simulated]"
```


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qtools.slm import SLM

slm = SLM(
    "simulated",
    resolution=(256, 192),
    pitch_um=6.4,
    wavelength=532e-9,
)
slm.connect()

slm.info


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/S15lib/g2lib/g2lib.py:17: UserWarning: Unable to import Cython conditional g2 module, using native option
  warnings.warn("Unable to import Cython conditional g2 module, using native option")


[INF] 15:46:54 SimulatedSLM Initialized SimulatedSLM.


{'shape': (192, 256),
 'pixel_pitch': (6.4, 6.4),
 'dimension': (1.6384, 1.2288000000000001),
 'wavelength_nm': 532.0,
 'bitdepth': 8,
 'backend': 'simulated',
 'model': 'slmsuite-SimulatedSLM',
 'headless': False}

In [2]:
canvas = slm.create_canvas()
print(canvas.info)

random_seed = 0
random_phase = canvas.random_phase(seed=random_seed)
# Reusing the seed creates the same random phase mask, which makes this test reproducible.
random_phase_repeat = canvas.random_phase(seed=random_seed)
assert np.array_equal(random_phase.array, random_phase_repeat.array)

masks = {
    "flat": canvas.flat(phase=np.pi / 2),
    "lens": canvas.lens(f=0.4),
    "binary": canvas.binary(period=16),
    "spiral": canvas.spiral(order=1),
    "perfect_spiral": canvas.perfect_spiral(order=1, radial_frequency=0.03),
    "blazed_grating": canvas.blazed_grating(
        period=16,
        angle=0,
        direction=1,
        order=None,
    ),
    "blazed_grating_8_levels": canvas.blazed_grating(
        period=16,
        angle=90,
        direction=-1,
        order=8,
    ),
    "zernike": canvas.zernike(index=4, radius=(110, 80)),
    "pseudo_knife_edge": canvas.pseudo_knife_edge(
        axis="x",
        edge=128,
        seed=0,
    ),
    "random": random_phase,
}

for name, phase in masks.items():
    assert phase.shape == slm.shape
    assert phase.geometry is slm.geometry
    assert np.isfinite(phase.array).all()
    print(f"{name:28s} {phase.info.get('phase_range_rad')}")

assert masks["blazed_grating"].info["period_pixels"] == 16
assert masks["blazed_grating"].info["phase_levels"] == 16
assert masks["blazed_grating_8_levels"].info["phase_levels"] == 8


{'pixels': (256, 192), 'resolution': (256, 192), 'shape': (192, 256), 'pixel_pitch': (6.4, 6.4), 'pixel_pitch_um': (6.4, 6.4), 'wavelength': 5.32e-07, 'wavelength_m': 5.32e-07, 'wavelength_nm': 532.0, 'bitdepth': 8, 'type': 'PhaseRegion', 'phase_range_rad': (0.0, 0.0), 'bounds': (0, 256, 0, 192), 'region_pixels': (256, 192)}
flat                         (1.5707963267948966, 1.5707963267948966)
lens                         (0.00030234876666127323, 15.345106954359604)
binary                       (0.0, 3.141592653589793)
spiral                       (-3.137671105065033, 3.137671105065033)
perfect_spiral               (-2.521804451219268, 32.52606779048551)
blazed_grating               (0.0, 5.890486225480862)
blazed_grating_8_levels      (0.0, 5.497787143782138)
zernike                      (-0.9998805526859504, 0.9999748192148761)
pseudo_knife_edge            (0.0, 6.28309914208202)
random                       (0.00014366467454098725, 6.283164991890043)


## Visualize all generated phase masks

The arrays are stored in radians. The display below wraps them to `[0, 2π)` only for visualization.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7), constrained_layout=True)
for axis, (name, phase) in zip(axes.flat, masks.items()):
    image = axis.imshow(np.mod(phase.array, 2 * np.pi), cmap="twilight", vmin=0, vmax=2 * np.pi)
    axis.set_title(name)
    axis.set_axis_off()
fig.colorbar(image, ax=axes, label="phase (rad)", shrink=0.8)
plt.show()


In [ ]:
# PhaseMask.show() uses inline output in Jupyter.
masks["blazed_grating"].show()
print(masks["blazed_grating"].info)


## Split-screen display of two phase masks

The left and right halves use independent masks. `compose()` merges them into one full SLM frame while preserving each mask only in its own region.


In [ ]:
left, right = canvas.partition(axis="x", ratios=(1, 1))
left_mask = left.spiral(order=1)
right_mask = right.blazed_grating(period=16, angle=0, direction=1)
split_phase = canvas.compose(left_mask, right_mask)

left_edge = slm.resolution[0] // 2
assert np.any(split_phase.array[:, :left_edge] != 0)
assert np.any(split_phase.array[:, left_edge:] != 0)
assert np.all(split_phase.array[:, :left_edge] == left_mask.array[:, :left_edge])
assert np.all(split_phase.array[:, left_edge:] == right_mask.array[:, left_edge:])

slm.load_phase(split_phase)
frame = slm.get_current_image()
assert frame is not None
assert frame.shape == (slm.shape[0], slm.shape[1], 3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for axis, title, values in zip(
    axes,
    ("left mask", "right mask", "split-screen composite"),
    (left_mask, right_mask, split_phase),
):
    axis.imshow(np.mod(values.array, 2 * np.pi), cmap="twilight", vmin=0, vmax=2 * np.pi)
    axis.set_title(title)
    axis.set_axis_off()
plt.show()


## Compose regional masks and load every phase into the simulated framebuffer


In [ ]:
left, right = canvas.partition(axis="x", ratios=(1, 1))
regional = canvas.compose(
    left.spiral(order=1),
    right.blazed_grating(period=16, angle=0, direction=1),
)
assert regional.shape == slm.shape
assert np.isfinite(regional.array).all()

for name, phase in {**masks, "regional_composite": regional}.items():
    display_mask = phase.to_display(bits=8)
    assert display_mask.resolution == slm.resolution
    slm.load(display_mask)
    frame = slm.get_current_image()
    assert frame is not None
    assert frame.shape == (slm.shape[0], slm.shape[1], 3)
    assert frame.dtype == np.uint8
    print(f"loaded {name:28s} framebuffer={frame.shape} dtype={frame.dtype}")

slm.disconnect()
print("simulated phase-mask notebook checks passed")
